# Field Operations Map

Application built for DSI Systems that accepts a data dump from Ossimview WebTracker and displays open work orders on an interactive map.

Jobs can be filtered by status and by project type (Starlink or DTV).

- Loads job data from a CSV export
- Geocodes ZIP codes to plot locations
- Clusters markers by status (Open, Confirmed, En Route, On Site, Scheduled, Needs Re/Scheduled, Callback, Other)
- Tree-layer control for filtering by DTV / Starlink and status

**Note**: This notebook expects a properly formatted CSV dump. Use anonymized or sample data for demonstration.


In [ ]:
#@title Field Ops Map
import geocoder
import folium
import pandas as pd
import warnings; warnings.filterwarnings('ignore')
from matplotlib import pyplot as plt
from folium.plugins import MarkerCluster
from folium.plugins.treelayercontrol import TreeLayerControl
from folium.features import Marker
%matplotlib inline

try:
  df = pd.read_csv('dump.csv')
except:
  print('Need to take a dump, but there is no dump to be found.')

df.dropna(subset=['City','State'],inplace=True)
df['geo'] = ' '

for i in df.index:
    try:
      g = geocoder.arcgis(df['ZIP'][i]).latlng
      df['geo'][i]=g
    except:
      df['geo'][i]='0,0'


m = folium.Map([37.0902, -95.7129], zoom_start=5)

#DTV map variables
dtv_popups = []
dtv_opn_popups = []
dtv_conf_popups = []
dtv_ons_popups = []
dtv_enr_popups = []
dtv_sch_popups = []
dtv_res_popups = []
dtv_cb_popups = []
dtv_oth_popups = []

dtv = pd.DataFrame(columns = df.columns)
dtv_opn = pd.DataFrame(columns = df.columns)
dtv_conf = pd.DataFrame(columns = df.columns)
dtv_ons = pd.DataFrame(columns = df.columns)
dtv_enr = pd.DataFrame(columns = df.columns)
dtv_sch = pd.DataFrame(columns = df.columns)
dtv_res = pd.DataFrame(columns = df.columns)
dtv_cb = pd.DataFrame(columns = df.columns)
dtv_oth = pd.DataFrame(columns = df.columns)

#Starlink map variables
stl_popups = []
stl_opn_popups = []
stl_sch_popups = []
stl_conf_popups = []
stl_res_popups = []
stl_enr_popups = []
stl_ons_popups = []
stl_cb_popups = []
stl_oth_popups = []

stl = pd.DataFrame(columns = df.columns)
stl_opn = pd.DataFrame(columns = df.columns)
stl_sch = pd.DataFrame(columns = df.columns)
stl_conf = pd.DataFrame(columns = df.columns)
stl_res = pd.DataFrame(columns = df.columns)
stl_enr = pd.DataFrame(columns = df.columns)
stl_ons = pd.DataFrame(columns = df.columns)
stl_cb = pd.DataFrame(columns = df.columns)
stl_oth = pd.DataFrame(columns = df.columns)

#Sort Starlink and DTV
for i in df.index:
    if 'Starlink' not in df['Project'][i]:
        dtv.loc[i] = df.loc[i]
        dtv_popups.append(str(dtv['TechOrderScheduled'][i])+'\n'+str(dtv['TechGroup'][i])+'\n'+str(dtv['Project'][i])+'\n'+str(dtv['AccountNumber'][i])+'\n'+str(dtv['LocationOrderStatus'][i]))

    if 'Starlink' in df['Project'][i]:
        stl.loc[i] = df.loc[i]
        stl_popups.append(str(stl['TechOrderScheduled'][i])+'\n'+str(stl['TechGroup'][i])+'\n'+str(stl['Project'][i])+'\n'+str(stl['AccountNumber'][i])+'\n'+str(stl['LocationOrderStatus'][i]))

#Sort DTV jobs by status
for i in dtv.index:
    if  dtv['LocationOrderStatus'][i] == 'Confirmed':
        dtv_conf.loc[i] = dtv.loc[i]
        dtv_conf_popups.append(str(dtv['TechOrderScheduled'][i])+'\n'+str(dtv['TechGroup'][i])+'\n'+str(dtv['Project'][i])+'\n'+str(dtv['AccountNumber'][i])+'\n'+str(dtv['LocationOrderStatus'][i]))

    elif  dtv['LocationOrderStatus'][i] == 'Open':
        dtv_opn.loc[i] = dtv.loc[i]
        dtv_opn_popups.append(str(dtv['TechOrderScheduled'][i])+'\n'+str(dtv['TechGroup'][i])+'\n'+str(dtv['Project'][i])+'\n'+str(dtv['AccountNumber'][i])+'\n'+str(dtv['LocationOrderStatus'][i]))

    elif  dtv['LocationOrderStatus'][i] == 'Tech En Route':
        dtv_enr.loc[i] = dtv.loc[i]
        dtv_enr_popups.append(str(dtv['TechOrderScheduled'][i])+'\n'+str(dtv['TechGroup'][i])+'\n'+str(dtv['Project'][i])+'\n'+str(dtv['AccountNumber'][i])+'\n'+str(dtv['LocationOrderStatus'][i]))

    elif  dtv['LocationOrderStatus'][i] == 'Tech On Site':
        dtv_ons.loc[i] = dtv.loc[i]
        dtv_ons_popups.append(str(dtv['TechOrderScheduled'][i])+'\n'+str(dtv['TechGroup'][i])+'\n'+str(dtv['Project'][i])+'\n'+str(dtv['AccountNumber'][i])+'\n'+str(dtv['LocationOrderStatus'][i]))

    elif  dtv['LocationOrderStatus'][i] == 'Scheduled':
        dtv_sch.loc[i] = dtv.loc[i]
        dtv_sch_popups.append(str(dtv['TechOrderScheduled'][i])+'\n'+str(dtv['TechGroup'][i])+'\n'+str(dtv['Project'][i])+'\n'+str(dtv['AccountNumber'][i])+'\n'+str(dtv['LocationOrderStatus'][i]))

    elif (dtv['LocationOrderStatus'][i] == 'Needs Scheduled') or (dtv['LocationOrderStatus'][i] == 'Needs Rescheduled'):
        dtv_res.loc[i] = dtv.loc[i]
        dtv_res_popups.append(str(dtv['TechOrderScheduled'][i])+'\n'+str(dtv['TechGroup'][i])+'\n'+str(dtv['Project'][i])+'\n'+str(dtv['AccountNumber'][i])+'\n'+str(dtv['LocationOrderStatus'][i]))

    elif  'Callback' in dtv['LocationOrderStatus'][i]:
        dtv_cb.loc[i] = dtv.loc[i]
        dtv_cb_popups.append(str(dtv['TechOrderScheduled'][i])+'\n'+str(dtv['TechGroup'][i])+'\n'+str(dtv['Project'][i])+'\n'+str(dtv['AccountNumber'][i])+'\n'+str(dtv['LocationOrderStatus'][i]))

    else:
        dtv_oth.loc[i] = dtv.loc[i]
        dtv_oth_popups.append(str(dtv['TechOrderScheduled'][i])+'\n'+str(dtv['TechGroup'][i])+'\n'+str(dtv['Project'][i])+'\n'+str(dtv['AccountNumber'][i])+'\n'+str(dtv['LocationOrderStatus'][i]))

#Sort Starlink jobs by status
for i in stl.index:
    if  stl['LocationOrderStatus'][i] == 'Confirmed':
        stl_conf.loc[i] = stl.loc[i]
        stl_conf_popups.append(str(stl['TechOrderScheduled'][i])+'\n'+str(stl['TechGroup'][i])+'\n'+str(stl['Project'][i])+'\n'+str(stl['AccountNumber'][i])+'\n'+str(stl['LocationOrderStatus'][i])+'\n'+str(stl['ConfirmationStatus'][i]))

    elif  stl['LocationOrderStatus'][i] == 'Open':
        stl_opn.loc[i] = stl.loc[i]
        stl_opn_popups.append(str(stl['TechOrderScheduled'][i])+'\n'+str(stl['TechGroup'][i])+'\n'+str(stl['Project'][i])+'\n'+str(stl['AccountNumber'][i])+'\n'+str(stl['LocationOrderStatus'][i])+'\n'+str(stl['ConfirmationStatus'][i]))

    elif  stl['LocationOrderStatus'][i] == 'Tech En Route':
        stl_enr.loc[i] = stl.loc[i]
        stl_enr_popups.append(str(stl['TechOrderScheduled'][i])+'\n'+str(stl['TechGroup'][i])+'\n'+str(stl['Project'][i])+'\n'+str(stl['AccountNumber'][i])+'\n'+str(stl['LocationOrderStatus'][i])+'\n'+str(stl['ConfirmationStatus'][i]))

    elif  stl['LocationOrderStatus'][i] == 'Tech On Site':
        stl_ons.loc[i] = stl.loc[i]
        stl_ons_popups.append(str(stl['TechOrderScheduled'][i])+'\n'+str(stl['TechGroup'][i])+'\n'+str(stl['Project'][i])+'\n'+str(stl['AccountNumber'][i])+'\n'+str(stl['LocationOrderStatus'][i])+'\n'+str(stl['ConfirmationStatus'][i]))

    elif  stl['LocationOrderStatus'][i] == 'Scheduled':
        stl_sch.loc[i] = stl.loc[i]
        stl_sch_popups.append(str(stl['TechOrderScheduled'][i])+'\n'+str(stl['TechGroup'][i])+'\n'+str(stl['Project'][i])+'\n'+str(stl['AccountNumber'][i])+'\n'+str(stl['LocationOrderStatus'][i])+'\n'+str(stl['ConfirmationStatus'][i]))

    elif (stl['LocationOrderStatus'][i] == 'Needs Scheduled') or (stl['LocationOrderStatus'][i] == 'Needs Rescheduled'):
        stl_res.loc[i] = stl.loc[i]
        stl_res_popups.append(str(stl['TechOrderScheduled'][i])+'\n'+str(stl['TechGroup'][i])+'\n'+str(stl['Project'][i])+'\n'+str(stl['AccountNumber'][i])+'\n'+str(stl['LocationOrderStatus'][i]))

    elif  'CALLBACK' in stl['LocationOrderStatus'][i].upper():
        stl_cb.loc[i] = stl.loc[i]
        stl_cb_popups.append(str(stl['TechOrderScheduled'][i])+'\n'+str(stl['TechGroup'][i])+'\n'+str(stl['Project'][i])+'\n'+str(stl['AccountNumber'][i])+'\n'+str(stl['LocationOrderStatus'][i]))

    else:
        stl_oth.loc[i] = stl.loc[i]
        stl_oth_popups.append(str(stl['TechOrderScheduled'][i])+'\n'+str(stl['TechGroup'][i])+'\n'+str(stl['Project'][i])+'\n'+str(stl['AccountNumber'][i])+'\n'+str(stl['LocationOrderStatus'][i]))

#DTV MarkerClusters
try:
    dtv_opn_mcg = MarkerCluster(locations=dtv_opn['geo'],popups=dtv_opn_popups,name='Open')
except ValueError:
    dtv_opn_mcg = Marker([0,0])

try:
    dtv_conf_mcg = MarkerCluster(locations=dtv_conf['geo'],popups=dtv_conf_popups,name='Confirmed')
except ValueError:
    dtv_conf_mcg = Marker([0,0])

try:
    dtv_enr_mcg = MarkerCluster(locations=dtv_enr['geo'],popups=dtv_enr_popups,name='Tech En Route')
except ValueError:
    dtv_enr_mcg = Marker([0,0])

try:
    dtv_ons_mcg = MarkerCluster(locations=dtv_ons['geo'],popups=dtv_ons_popups,name='Tech On Site')
except ValueError:
    dtv_ons_mcg = Marker([0,0])

try:
    dtv_sch_mcg = MarkerCluster(locations=dtv_sch['geo'],popups=dtv_sch_popups,name='Scheduled')
except ValueError:
    dtv_sch_mcg = Marker([0,0])

try:
    dtv_res_mcg = MarkerCluster(locations=dtv_res['geo'],popups=dtv_res_popups,name='Needs Re/Scheduled')
except ValueError:
    dtv_res_mcg = Marker([0,0])

try:
    dtv_cb_mcg = MarkerCluster(locations=dtv_cb['geo'],popups=dtv_cb_popups,name='Callback')
except ValueError:
    dtv_cb_mcg = Marker([0,0])

try:
    dtv_oth_mcg = MarkerCluster(locations=dtv_oth['geo'],popups=dtv_oth_popups,name='Other')
except ValueError:
    dtv_oth_mcg = Marker([0,0])

#Starlink MarkerClusters
try:
    stl_opn_mcg = MarkerCluster(locations=stl_opn['geo'],popups=stl_opn_popups,name='Open')
except ValueError:
    stl_opn_mcg = Marker([0,0])

try:
    stl_conf_mcg = MarkerCluster(locations=stl_conf['geo'],popups=stl_conf_popups,name='Confirmed')
except ValueError:
    stl_conf_mcg = Marker([0,0])

try:
    stl_enr_mcg = MarkerCluster(locations=stl_enr['geo'],popups=stl_enr_popups,name='Tech En Route')
except ValueError:
    stl_enr_mcg = Marker([0,0])

try:
    stl_ons_mcg = MarkerCluster(locations=stl_ons['geo'],popups=stl_ons_popups,name='Tech On Site')
except ValueError:
    stl_ons_mcg = Marker([0,0])

try:
    stl_sch_mcg = MarkerCluster(locations=stl_sch['geo'],popups=stl_sch_popups,name='Scheduled')
except ValueError:
    stl_sch_mcg = Marker([0,0])

try:
    stl_res_mcg = MarkerCluster(locations=stl_res['geo'],popups=stl_res_popups,name='Needs Re/Scheduled')
except ValueError:
    stl_res_mcg = Marker([0,0])

try:
    stl_cb_mcg = MarkerCluster(locations=stl_cb['geo'],popups=stl_cb_popups,name='Callback')
except ValueError:
    stl_cb_mcg = Marker([0,0])

try:
    stl_oth_mcg = MarkerCluster(locations=stl_oth['geo'],popups=stl_oth_popups,name='Other')
except ValueError:
    stl_oth_mcg = Marker([0,0])

overlay_tree = {
    "label": "Field Ops",
    "select_all_checkbox": "Un/select all",
    "children": [
        {
            "label": "DTV",
            "select_all_checkbox": True,
            "children": [
                        { "label": "Open", "layer": : dtv_opn_mcg.add_to(m) },
                        { "label": "Confirmed", "layer": : dtv_conf_mcg.add_to(m) },
                        { "label": "En Route", "layer": : dtv_enr_mcg.add_to(m) },
                        { "label": "On Site", "layer": : dtv_ons_mcg.add_to(m) },
                        { "label": "Scheduled", "layer": : dtv_sch_mcg.add_to(m) },
                        { "label": "Needs Re/Scheduled", "layer": : dtv_res_mcg.add_to(m) },
                        { "label": "Callback", "layer": : dtv_cb_mcg.add_to(m) },
                        { "label": "Other", "layer": : dtv_oth_mcg.add_to(m) }
                    ]
                },
        {
            "label": "Starlink",
            "select_all_checkbox": True,
            "children": [
                        { "label": "Open", "layer": : stl_opn_mcg.add_to(m) },
                        { "label": "Confirmed", "layer": : stl_conf_mcg.add_to(m) },
                        { "label": "En Route", "layer": : stl_enr_mcg.add_to(m) },
                        { "label": "On Site", "layer": : stl_ons_mcg.add_to(m) },
                        { "label": "Scheduled", "layer": : stl_sch_mcg.add_to(m) },
                        { "label": "Needs Re/Scheduled", "layer": : stl_res_mcg.add_to(m) },
                        { "label": "Callback", "layer": : stl_cb_mcg.add_to(m) },
                        { "label": "Other", "layer": : stl_oth_mcg.add_to(m) }
            ]
        }
    ]
}


TreeLayerControl(overlay_tree=overlay_tree).add_to(m)
m

In [ ]:
#@title Last Updated
from datetime import date
print(date.today())